In [1]:
import pandas as pd                 # 표 형태 데이터를 다루는 라이브러리
import numpy as np                  # 수치 계산용 라이브러리 (RMSE 계산 등에 사용)
import matplotlib.pyplot as plt     # 그래프 시각화 라이브러리

# train_test_split : 데이터를 학습용과 검즈용으로 나누는 함수
from sklearn.model_selection import train_test_split
# 회귀 모델 평가에 사용할 지표들 (MAE, MSE->RMSE, R2)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# ColumnTransformer: 컬럼 종류별로 다른 전처리를 적용하는 도구
from sklearn.compose import ColumnTransformer
# Pipeline: 전처리와 모델 학습을 하나의 흐름으로 묶는 도구
from sklearn.pipeline import Pipeline
# OneHotEncoder: 문자형(범주형) 값을 숫자 형태로 바꾸는 도구
from sklearn.preprocessing import OneHotEncoder
# SimpleImputer: 비어 있는 값(결측치)을 정해진 규칙으로 채우는 도구
from sklearn.impute import SimpleImputer

# XGBRegressor: 숫자 값을 예측하는 회귀용 XGBoost 모델
from xgboost import XGBRegressor

In [2]:
# 그래프 모양 설정
plt.style.use("ggplot")

In [3]:
import matplotlib.font_manager as fm
import warnings

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
warnings.filterwarnings("ignore", category=UserWarning)

In [4]:
df = pd.read_csv("jeju_bus.csv")
# df_model = df.copy()

In [5]:
df.head()

,id,date,route_id,vh_id,route_nm,now_latitude,now_longitude,now_station,now_arrive_time,distance,next_station,next_latitude,next_longitude,next_arrive_time
0,0,2019-10-15,405136001,7997025,360-1,33.456267,126.551750,제주대학교입구,06시,266.0,제대마을,33.457724,126.554014,24
1,1,2019-10-15,405136001,7997025,360-1,33.457724,126.554014,제대마을,06시,333.0,제대아파트,33.458783,126.557353,36
2,2,2019-10-15,405136001,7997025,360-1,33.458783,126.557353,제대아파트,06시,415.0,제주대학교,33.459893,126.561624,40
3,3,2019-10-15,405136001,7997025,360-1,33.479705,126.543811,남국원(아라방면),06시,578.0,제주여자중고등학교(아라방면),33.484860,126.542928,42
4,4,2019-10-15,405136001,7997025,360-1,33.485662,126.494923,도호동,07시,374.0,은남동,33.485822,126.490897,64


In [6]:
df.shape

(210457, 14)

In [7]:
df.columns

Index(['id', 'date', 'route_id', 'vh_id', 'route_nm', 'now_latitude',
       'now_longitude', 'now_station', 'now_arrive_time', 'distance',
       'next_station', 'next_latitude', 'next_longitude', 'next_arrive_time'],
      dtype='str')

In [8]:
df.isnull().sum()

id                  0
date                0
route_id            0
vh_id               0
route_nm            0
now_latitude        0
now_longitude       0
now_station         0
now_arrive_time     0
distance            0
next_station        0
next_latitude       0
next_longitude      0
next_arrive_time    0
dtype: int64

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 210457 entries, 0 to 210456
Data columns (total 14 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                210457 non-null  int64  
 1   date              210457 non-null  str    
 2   route_id          210457 non-null  int64  
 3   vh_id             210457 non-null  int64  
 4   route_nm          210457 non-null  str    
 5   now_latitude      210457 non-null  float64
 6   now_longitude     210457 non-null  float64
 7   now_station       210457 non-null  str    
 8   now_arrive_time   210457 non-null  str    
 9   distance          210457 non-null  float64
 10  next_station      210457 non-null  str    
 11  next_latitude     210457 non-null  float64
 12  next_longitude    210457 non-null  float64
 13  next_arrive_time  210457 non-null  int64  
dtypes: float64(5), int64(4), str(5)
memory usage: 22.5 MB


In [10]:
df.describe()

,id,route_id,vh_id,now_latitude,now_longitude,distance,next_latitude,next_longitude,next_arrive_time
count,210457.000000,2.104570e+05,2.104570e+05,210457.000000,210457.000000,210457.000000,210457.000000,210457.000000,210457.000000
mean,105228.000000,4.052491e+08,7.988694e+06,33.434528,126.603451,490.256100,33.434711,126.603687,85.380824
std,60753.847139,9.132404e+04,6.774077e+03,0.102350,0.123961,520.563932,0.102224,0.123838,85.051170
min,0.000000,4.051360e+08,7.983000e+06,33.244382,126.473300,97.000000,33.244382,126.473300,6.000000
25%,52614.000000,4.051365e+08,7.983093e+06,33.325283,126.523900,291.000000,33.325283,126.524550,44.000000
50%,105228.000000,4.053201e+08,7.983431e+06,33.484667,126.551050,384.000000,33.484860,126.551050,66.000000
75%,157842.000000,4.053201e+08,7.997041e+06,33.500197,126.650322,542.000000,33.500228,126.650322,102.000000
max,210456.000000,4.053281e+08,7.997124e+06,33.556167,126.935188,7461.000000,33.556167,126.935188,2996.000000


target_col = df.columns[-1]

target_col

df.columns

required_cols = df.columns[1:-1]

required_cols

data = df.copy()

data

data["date"]

data["date"] = pd.to_datetime(data["date"])

data["date"].dt.year

data["date"].dt.month

data["date"].dt.dayofweek

data["year"] = data["date"].dt.year

data["month"] = data["date"].dt.month

data

data["now_arrive_time"]

data["now_hour"] = data["now_arrive_time"].str.extract(r"(\d+)").astype(float)

data

data["now_hour"]

data.columns

for col in required_cols:
    print(f"col={col}")
    print("="*100)

[col for col in required_cols if col not in ["date", "vh_id"]]

required_cols[:-2]

missing_cols = [col for col in required_cols if col not in required_cols[:-2]]

missing_cols

True + True + False

if missing_cols:
    print("missing cols 는 True!@#$")
else:
    print("missing cols 는 False")

if missing_cols:
    raise ValueError(f"필수 컬럼 누락!! : {missing_cols}")

missing_cols = [col for col in required_cols if col not in data.columns]

if missing_cols:
    print("missing_cols True!!")
else:
    print("missing_cols False..")

data.columns

target_col

if target_col in data.columns:
    print("target_col true")
else:
    print("target_col false")

snack_df = pd.DataFrame({
    "과자명":["초코파이","오예스","고북칩","짱구"]
})

snack_df

#OneHot Encoding 할꺼다(호소중)
categorical_transformer = Pipeline(steps=[("onehot", OneHotEncoder())])

#과자명 컬럼 One Hot Encoding 할꺼다
preprocessor = ColumnTransformer(
    transformers=[("cat", categorical_transformer,["과자명"])]
)

preprocessor.fit_transform(snack_df)

pd.DataFrame(
    preprocessor.fit_transform(snack_df).toarray(),
    columns = preprocessor.get_feature_names_out()
)

X = prepare_features(df_model)

X

In [11]:
# 예측 대상(정답) 컬럼명을 변수로 지정해 둡니다. 이후 코드에서 이 변수를 일관되게 사용합니다.
target_col = "next_arrive_time"

# 전처리 입력에 반드시 있어야 하는 필수 컬럼 목록입니다.
# 이 목록은 아래 prepare_features() 안에서 "누락된 컬럼이 없는지" 검사하는 데 사용됩니다.
# 실제 데이터 컬럼명이 다르면 df.columns 출력을 확인한 뒤 이 목록을 수정합니다.
required_cols = [
    "route_id",
    "vh_id",
    "route_nm",
    "now_latitude",
    "now_longitude",
    "now_station",
    "now_arrive_time",
    "distance",
    "next_station",
    "next_latitude",
    "next_longitude",
    "date"
]

In [12]:
def prepare_features(df_input):

    # 원본을 직접 수정하지 않도록 복사본을 만들어 작업합니다.
    # (함수 밖의 데이터가 의도치 않게 바뀌는 것을 막기 위함입니다.)
    data = df_input.copy()

    # 필수 컬럼이 빠지지 않았는지 먼저 검사합니다.
    # 누락된 컬럼이 있으면 어떤 컬럼이 없는지 알려 주는 오류를 발생시켜 원인을 빨리 찾게 합니다.
    missing_cols = [col for col in required_cols if col not in data.columns]
    if missing_cols:
        raise ValueError(f"필수 컬럼이 누락되었습니다: {missing_cols}")

    # date를 datetime(날짜형)으로 변환한 뒤, 모델이 활용하기 좋은 숫자 파생 컬럼을 만듭니다.
    data["date"] = pd.to_datetime(data["date"])
    data["year"] = data["date"].dt.year                  # 연도
    data["month"] = data["date"].dt.month                # 월
    data["day"] = data["date"].dt.day                    # 일
    data["dayofweek"] = data["date"].dt.dayofweek        # 요일 (월=0 ... 일=6)

    # now_arrive_time의 "06시" 같은 문자열에서 숫자 부분만 추출해 now_hour(정수 시간)로 만듭니다.
    # str.extract(r"(\d+)")는 문자열에서 연속된 숫자를 뽑아내는 처리입니다. ("06시" -> "06")
    data["now_hour"] = (
        data["now_arrive_time"].astype(str).str.extract(r"(\d+)").astype(float)
    )

    # 변환을 마친 원본 date, now_arrive_time 컬럼은 더 이상 필요 없으므로 feature에서 제외합니다.
    data = data.drop(columns=["date", "now_arrive_time"])

    # 정답(target) 컬럼이 들어 있으면 제거합니다. (입력 feature에는 정답이 포함되면 안 됩니다.)
    # 예측용 데이터에는 보통 target이 없으므로, 있을 때만 제거하도록 조건을 둡니다.
    if target_col in data.columns:
        data = data.drop(columns=[target_col])

    # 단순 식별 번호인 id 컬럼이 있으면 제거합니다. (예측에 도움이 되지 않는 정보입니다.)
    if "id" in data.columns:
        data = data.drop(columns=["id"])

    return data  # 정리된 feature DataFrame을 돌려줍니다.

In [13]:
df_model = df.copy()

X_preprocess = prepare_features(df_model)

In [14]:
X_preprocess

,route_id,vh_id,route_nm,now_latitude,now_longitude,now_station,distance,next_station,next_latitude,next_longitude,year,month,day,dayofweek,now_hour
0,405136001,7997025,360-1,33.456267,126.551750,제주대학교입구,266.0,제대마을,33.457724,126.554014,2019,10,15,1,6.0
1,405136001,7997025,360-1,33.457724,126.554014,제대마을,333.0,제대아파트,33.458783,126.557353,2019,10,15,1,6.0
2,405136001,7997025,360-1,33.458783,126.557353,제대아파트,415.0,제주대학교,33.459893,126.561624,2019,10,15,1,6.0
3,405136001,7997025,360-1,33.479705,126.543811,남국원(아라방면),578.0,제주여자중고등학교(아라방면),33.484860,126.542928,2019,10,15,1,6.0
4,405136001,7997025,360-1,33.485662,126.494923,도호동,374.0,은남동,33.485822,126.490897,2019,10,15,1,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
210452,405328102,7983486,281-2,33.255783,126.577450,비석거리,528.0,삼아아파트,33.251896,126.574417,2019,10,28,0,21.0
210453,405328102,7983486,281-2,33.248595,126.568527,동문로터리,280.0,매일올레시장 7번입구,33.249753,126.565959,2019,10,28,0,21.0
210454,405328102,7983486,281-2,33.251891,126.560303,서귀포시 구 버스터미널,114.0,아랑조을거리 입구,33.251084,126.559551,2019,10,28,0,21.0
210455,405328102,7983486,281-2,33.251084,126.559551,아랑조을거리 입구,223.0,평생학습관,33.249504,126.558068,2019,10,28,0,21.0


In [15]:
categorical_features = [
    "route_nm",
    "now_station",
    "next_station"
]

categorical_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("cat", categorical_transformer, categorical_features)
], remainder = "passthrough")

In [16]:
X_cat_encoded = preprocessor.fit_transform(X_preprocess)

In [28]:
encoded_feature_names = preprocessor.get_feature_names_out()
encoded_feature_names

array(['cat__route_nm_201-11', 'cat__route_nm_201-12',
       'cat__route_nm_201-13', 'cat__route_nm_201-14',
       'cat__route_nm_201-15', 'cat__route_nm_201-16',
       'cat__route_nm_201-17', 'cat__route_nm_201-18',
       'cat__route_nm_201-21', 'cat__route_nm_201-22',
       'cat__route_nm_201-24', 'cat__route_nm_201-26',
       'cat__route_nm_201-27', 'cat__route_nm_281-1',
       'cat__route_nm_281-2', 'cat__route_nm_360-1',
       'cat__route_nm_360-12', 'cat__route_nm_360-2',
       'cat__route_nm_360-7', 'cat__route_nm_365-21',
       'cat__route_nm_365-22', 'cat__now_station_911의원',
       'cat__now_station_LH아파트', 'cat__now_station_가마초등학교',
       'cat__now_station_가흥동', 'cat__now_station_거로 입구',
       'cat__now_station_견월교', 'cat__now_station_계룡동',
       'cat__now_station_고도농원', 'cat__now_station_고래왓',
       'cat__now_station_고망난돌입구', 'cat__now_station_고산동산(광양방면)',
       'cat__now_station_고산동산(아라방면)', 'cat__now_station_고성리 구 성산농협',
       'cat__now_station_고성리 성산농협', 

In [18]:
X_cat_encoded_df = pd.DataFrame(
    X_cat_encoded.toarray(),
    columns = encoded_feature_names
)

In [19]:
X_cat_encoded_df

,cat__route_nm_201-11,cat__route_nm_201-12,cat__route_nm_201-13,cat__route_nm_201-14,cat__route_nm_201-15,cat__route_nm_201-16,cat__route_nm_201-17,cat__route_nm_201-18,cat__route_nm_201-21,cat__route_nm_201-22,...,remainder__now_latitude,remainder__now_longitude,remainder__distance,remainder__next_latitude,remainder__next_longitude,remainder__year,remainder__month,remainder__day,remainder__dayofweek,remainder__now_hour
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,33.456267,126.551750,266.0,33.457724,126.554014,2019.0,10.0,15.0,1.0,6.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,33.457724,126.554014,333.0,33.458783,126.557353,2019.0,10.0,15.0,1.0,6.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,33.458783,126.557353,415.0,33.459893,126.561624,2019.0,10.0,15.0,1.0,6.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,33.479705,126.543811,578.0,33.484860,126.542928,2019.0,10.0,15.0,1.0,6.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,33.485662,126.494923,374.0,33.485822,126.490897,2019.0,10.0,15.0,1.0,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
210452,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,33.255783,126.577450,528.0,33.251896,126.574417,2019.0,10.0,28.0,0.0,21.0
210453,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,33.248595,126.568527,280.0,33.249753,126.565959,2019.0,10.0,28.0,0.0,21.0
210454,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,33.251891,126.560303,114.0,33.251084,126.559551,2019.0,10.0,28.0,0.0,21.0
210455,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,33.251084,126.559551,223.0,33.249504,126.558068,2019.0,10.0,28.0,0.0,21.0


In [20]:
y = df_model[target_col]

y

0         24
1         36
2         40
3         42
4         64
          ..
210452    96
210453    50
210454    16
210455    38
210456    24
Name: next_arrive_time, Length: 210457, dtype: int64

In [21]:
new_required_cols = [
    "route_id",
    "vh_id",
    "route_nm",
    "now_latitude",
    "now_longitude",
    "now_station",
    "distance",
    "next_station",
    "next_latitude",
    "next_longitude",
    "year",
    "month",
    "day",
    "dayofweek",
    "now_hour"
]

In [22]:
X = X_preprocess[new_required_cols]
X

,route_id,vh_id,route_nm,now_latitude,now_longitude,now_station,distance,next_station,next_latitude,next_longitude,year,month,day,dayofweek,now_hour
0,405136001,7997025,360-1,33.456267,126.551750,제주대학교입구,266.0,제대마을,33.457724,126.554014,2019,10,15,1,6.0
1,405136001,7997025,360-1,33.457724,126.554014,제대마을,333.0,제대아파트,33.458783,126.557353,2019,10,15,1,6.0
2,405136001,7997025,360-1,33.458783,126.557353,제대아파트,415.0,제주대학교,33.459893,126.561624,2019,10,15,1,6.0
3,405136001,7997025,360-1,33.479705,126.543811,남국원(아라방면),578.0,제주여자중고등학교(아라방면),33.484860,126.542928,2019,10,15,1,6.0
4,405136001,7997025,360-1,33.485662,126.494923,도호동,374.0,은남동,33.485822,126.490897,2019,10,15,1,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
210452,405328102,7983486,281-2,33.255783,126.577450,비석거리,528.0,삼아아파트,33.251896,126.574417,2019,10,28,0,21.0
210453,405328102,7983486,281-2,33.248595,126.568527,동문로터리,280.0,매일올레시장 7번입구,33.249753,126.565959,2019,10,28,0,21.0
210454,405328102,7983486,281-2,33.251891,126.560303,서귀포시 구 버스터미널,114.0,아랑조을거리 입구,33.251084,126.559551,2019,10,28,0,21.0
210455,405328102,7983486,281-2,33.251084,126.559551,아랑조을거리 입구,223.0,평생학습관,33.249504,126.558068,2019,10,28,0,21.0


In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train.shape, X_test.shape

((168365, 15), (42092, 15))

In [24]:
xgb_model = XGBRegressor()

model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", xgb_model)
])

_ = model_pipeline.fit(X_train, y_train)

print("모델 학습 완료")

모델 학습 완료


In [25]:
y_pred = model_pipeline.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)

print(f"MAE: {mae:.4f}")

MAE: 23.0865


In [26]:
# 예측할 새로운 데이터
new_data = pd.DataFrame([
    {
        "date": "2019-10-21",
        "route_id": 405136001,
        "vh_id": 7997025,
        "route_nm": "360-1",
        "now_latitude": 33.456267,
        "now_longitude": 126.551750,
        "now_station": "제주대학교입구",
        "now_arrive_time": "08시",
        "distance": 300.0,
        "next_station": "제대마을",
        "next_latitude": 33.457724,
        "next_longitude": 126.554014
    }
])

new_data

,date,route_id,vh_id,route_nm,now_latitude,now_longitude,now_station,now_arrive_time,distance,next_station,next_latitude,next_longitude
0,2019-10-21,405136001,7997025,360-1,33.456267,126.55175,제주대학교입구,08시,300.0,제대마을,33.457724,126.554014


In [27]:
new_data_prepared = prepare_features(new_data)

new_pred = model_pipeline.predict(new_data_prepared)

result_df = new_data.copy()
result_df["predicted_next_arrive_time"] = new_pred
result_df["predicted_next_arrive_time"] = result_df["predicted_next_arrive_time"].round(2)

result_df[[
    "route_nm",
    "now_station",
    "next_station",
    "now_arrive_time",
    "distance",
    "predicted_next_arrive_time"
]]

,route_nm,now_station,next_station,now_arrive_time,distance,predicted_next_arrive_time
0,360-1,제주대학교입구,제대마을,08시,300.0,37.27
